# BaddieVision - Single-Video Feature Extraction

This Colab notebook extracts reusable artifacts for exactly one input video:

- `tracks.csv` for shuttle detections
- `players.csv` for the current heuristic player-activity schema
- `player_poses.jsonl` as the raw player tracking + pose source of truth
- `metadata.json` with video and runtime details
- `preview.mp4` with shuttle and player overlays
- `<source_id>.zip` containing the full output folder

It does **not** run rally segmentation or emit rally intervals. The goal is one extraction pass on Colab so local processing can be rerun later without GPU work.


## Runtime Notes

Use **Runtime > Change runtime type > T4 GPU** (or another GPU).

The notebook expects the private model bundle ZIP to be available in Google Drive under `MyDrive/BaddieVision/`. By default it uses one uploaded video, but it can also read a Drive path directly.


## 1. Configure the run

Set the input mode and optional source ID. Keep `PERSON_YOLO_MODEL` on a COCO person model such as `yolov8n.pt`; shuttle YOLO weights are only for shuttle detection and are not used for player tracking.


In [ ]:
from pathlib import Path

GITHUB_URL = "https://github.com/MaxLinCode/BaddieVision.git"
REPO_DIR = Path("/content/BaddieVision")
DRIVE_ROOT = Path("/content/drive/MyDrive/BaddieVision")

# Required model bundle in Drive.
MODELS_ZIP_NAME = "badminton-models-v1-2026-07-01.zip"

# Input mode: "upload" or "drive".
INPUT_MODE = "upload"
DRIVE_VIDEO_PATH = ""  # Example: /content/drive/MyDrive/BaddieVision/input/my_match.mp4

# Optional overrides.
SOURCE_ID = ""
# COCO person detector for player tracking. Do not use the shuttle-trained YOLO here.
PERSON_YOLO_MODEL = "yolov8n.pt"
COURT_CALIBRATION = ""  # Recommended for skewed/sideways cameras; without it, player selection falls back to a rough screen-space heuristic.
# TrackNet background estimation can use a lot of RAM. Lower values reduce memory use.
TRACKNET_MAX_SAMPLE_NUM = 240
TRACKNET_VIDEO_RANGE = None  # Example: (0, 45) to estimate the median from only the first 45 seconds.
OUTPUT_ROOT = Path("/content/colab_outputs")
COPY_ZIP_TO_DRIVE = True
DOWNLOAD_ZIP_TO_BROWSER = False

print(f"Repository target: {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Input mode: {INPUT_MODE}")


## 2. Mount Drive

Drive is used for the private model bundle and optional ZIP export.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone and install

This keeps the notebook independent from the current Colab filesystem state.


In [ ]:
import subprocess

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"], check=True)
else:
    subprocess.run(["git", "clone", "--recurse-submodules", GITHUB_URL, str(REPO_DIR)], check=True)

print(f"Repository ready at {REPO_DIR}")


In [ ]:
%cd /content/BaddieVision
%pip install -q -r requirements.txt


## 4. Load imports

These imports also put the repo and TrackNet submodule on `sys.path`.


In [ ]:
import csv
import json
import os
import shutil
import sys
import zipfile
from datetime import datetime, timezone

import cv2
import mediapipe as mp
import torch

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

tracknet_dir = src_dir / "TrackNetV3"
if str(tracknet_dir) not in sys.path:
    sys.path.insert(0, str(tracknet_dir))

from InPlay.heuristic.players import crop_point_to_image, run as run_player_extraction
from InPlay.heuristic.tracks import read_track_csv
from InPlay.heuristic.config import HeuristicConfig
from TrackNetV3.predictArgs import load_torchscript_model, run_prediction


## 5. Input helpers

Restore the model bundle and resolve exactly one input video.


In [ ]:
def extract_models_zip() -> None:
    archive = DRIVE_ROOT / MODELS_ZIP_NAME
    if not archive.is_file():
        raise FileNotFoundError(f"Missing model bundle: {archive}")
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted {archive.name} into {REPO_DIR}")


def resolve_input_video() -> Path:
    if INPUT_MODE == "drive":
        if not DRIVE_VIDEO_PATH:
            raise ValueError("Set DRIVE_VIDEO_PATH when INPUT_MODE='drive'.")
        video_path = Path(DRIVE_VIDEO_PATH)
        if not video_path.is_file():
            raise FileNotFoundError(video_path)
        return video_path
    if INPUT_MODE != "upload":
        raise ValueError("INPUT_MODE must be 'upload' or 'drive'.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one uploaded video, got {len(uploaded)}")
    filename = next(iter(uploaded))
    return Path("/content") / filename


## 6. Model and video helpers

Load TrackNet/InpaintNet and read source video properties for validation and metadata.


In [ ]:
def load_track_models() -> dict[str, object]:
    tracknet_script = REPO_DIR / "models" / "TrackNet_torchscript.pt"
    inpaintnet_script = REPO_DIR / "models" / "InpaintNet_torchscript.pt"
    tracknet_ckpt = torch.load(REPO_DIR / "src" / "TrackNetV3" / "ckpts" / "TrackNet_best.pt", map_location=torch.device("cpu"))
    inpaintnet_ckpt = torch.load(REPO_DIR / "src" / "TrackNetV3" / "ckpts" / "InpaintNet_best.pt", map_location=torch.device("cpu"))
    return {
        "tracknet": load_torchscript_model(str(tracknet_script)),
        "inpaintnet": load_torchscript_model(str(inpaintnet_script)),
        "tracknet_seq_len": int(tracknet_ckpt["param_dict"]["seq_len"]),
        "inpaintnet_seq_len": int(inpaintnet_ckpt["param_dict"]["seq_len"]),
        "bg_mode": tracknet_ckpt["param_dict"]["bg_mode"],
        "tracknet_checkpoint": str(REPO_DIR / "src" / "TrackNetV3" / "ckpts" / "TrackNet_best.pt"),
        "inpaintnet_checkpoint": str(REPO_DIR / "src" / "TrackNetV3" / "ckpts" / "InpaintNet_best.pt"),
    }


def read_video_info(path: Path) -> dict[str, object]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {path}")
    info = {
        "fps": float(cap.get(cv2.CAP_PROP_FPS) or 0.0),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return info


## 7. Preview helpers

Render a QA video from `tracks.csv` plus the raw player pose artifact.


In [ ]:
def load_track_rows(path: Path) -> dict[int, dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return {int(row["Frame"]): row for row in csv.DictReader(handle)}


def draw_pose_landmarks(frame, bbox, pose_landmarks):
    if not pose_landmarks:
        return
    points = []
    for item in pose_landmarks:
        x, y = crop_point_to_image((item["x"], item["y"]), bbox)
        points.append((int(round(x)), int(round(y)), float(item.get("visibility", 0.0))))
    for start, end in mp.solutions.pose.POSE_CONNECTIONS:
        if start >= len(points) or end >= len(points):
            continue
        ax, ay, av = points[start]
        bx, by, bv = points[end]
        if av < 0.5 or bv < 0.5:
            continue
        cv2.line(frame, (ax, ay), (bx, by), (255, 180, 0), 2, cv2.LINE_AA)
    for x, y, visibility in points:
        if visibility >= 0.5:
            cv2.circle(frame, (x, y), 2, (0, 255, 255), -1)


def render_preview(video_path: Path, tracks_csv: Path, player_raw_jsonl: Path, output_path: Path) -> None:
    track_rows = load_track_rows(tracks_csv)
    cap = cv2.VideoCapture(str(video_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    frame_index = 0
    with player_raw_jsonl.open(encoding="utf-8") as handle:
        metadata = json.loads(next(handle))
        if metadata.get("type") != "metadata":
            raise ValueError("player raw artifact must start with a metadata record")
        for raw_line in handle:
            ok, frame = cap.read()
            if not ok:
                break
            player_frame = json.loads(raw_line)
            if player_frame.get("type") != "frame":
                continue
            frame_index = int(player_frame["frame"])
            track = track_rows.get(frame_index)
            if track and int(track.get("Visibility", 0)) == 1:
                x = int(float(track["X"]))
                y = int(float(track["Y"]))
                cv2.circle(frame, (x, y), 7, (0, 255, 0), 2)
                cv2.putText(frame, "Shuttle", (x + 8, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
            for detection in player_frame.get("detections", []):
                bbox = detection.get("bbox")
                if not bbox:
                    continue
                x1, y1, x2, y2 = [int(round(value)) for value in bbox]
                color = (0, 200, 255) if detection.get("selected") else (120, 120, 120)
                thickness = 2 if detection.get("selected") else 1
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
                label = f"ID {int(detection['track_id'])}"
                if detection.get("slot"):
                    label = f"P{int(detection['slot'])} {label}"
                if detection.get("activity") is not None:
                    label += f" act {float(detection['activity']):.3f}"
                cv2.putText(frame, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
                draw_pose_landmarks(frame, bbox, detection.get("pose_landmarks"))
            writer.write(frame)
    cap.release()
    writer.release()
    print(f"Preview written to {output_path}")


## 8. Metadata and ZIP helpers

Write the extraction contract and package the output folder.


In [ ]:
def write_metadata(path: Path, source_id: str, video_path: Path, video_info: dict[str, object], models: dict[str, object]) -> None:
    metadata = {
        "source_id": source_id,
        "original_filename": video_path.name,
        "fps": video_info["fps"],
        "width": video_info["width"],
        "height": video_info["height"],
        "frame_count": video_info["frame_count"],
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model_info": {
            "tracknet_checkpoint": models["tracknet_checkpoint"],
            "inpaintnet_checkpoint": models["inpaintnet_checkpoint"],
            "player_detector": PERSON_YOLO_MODEL,
            "player_detector_class_id": 0,
            "player_detector_class_name": "person",
            "pose_model": "mediapipe.solutions.pose.Pose",
        },
        "runtime_info": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        },
    }
    path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Metadata written to {path}")


def zip_result_dir(result_dir: Path, zip_path: Path) -> None:
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(result_dir.rglob("*")):
            if file_path.is_file() and file_path != zip_path:
                zf.write(file_path, arcname=file_path.relative_to(result_dir.parent))
    print(f"ZIP written to {zip_path}")


## 9. Prepare output folder

This creates `colab_outputs/<video_stem>/` and sets the default `source_id`.


In [ ]:
extract_models_zip()
video_path = resolve_input_video()
source_id = SOURCE_ID or video_path.stem
result_dir = OUTPUT_ROOT / video_path.stem
result_dir.mkdir(parents=True, exist_ok=True)

local_video_path = video_path

print(f"Video ready at {local_video_path}")
print(f"Source ID: {source_id}")
print(f"Result dir: {result_dir}")


## 10. Run extraction

This is the main GPU work: shuttle extraction, player extraction, validation, preview rendering, metadata, and ZIP creation. It does not run rally segmentation.


In [ ]:
video_info = read_video_info(local_video_path)
models = load_track_models()

tracks_path = result_dir / "tracks.csv"
players_path = result_dir / "players.csv"
player_raw_path = result_dir / "player_poses.jsonl"
metadata_path = result_dir / "metadata.json"
preview_path = result_dir / "preview.mp4"
zip_path = result_dir / f"{source_id}.zip"

run_prediction(
    video_file=str(local_video_path),
    save_dir=str(result_dir),
    batch_size=4,
    output_video=False,
    large_video=True,
    max_sample_num=TRACKNET_MAX_SAMPLE_NUM,
    video_range=TRACKNET_VIDEO_RANGE,
    reuse_models={
        "tracknet": models["tracknet"],
        "inpaintnet": models["inpaintnet"],
        "tracknet_seq_len": models["tracknet_seq_len"],
        "inpaintnet_seq_len": models["inpaintnet_seq_len"],
        "bg_mode": models["bg_mode"],
    },
)

tracknet_csv = result_dir / f"{local_video_path.stem.lower()}_ball.csv"
if not tracknet_csv.is_file():
    raise FileNotFoundError(f"TrackNet output missing: {tracknet_csv}")
shutil.move(tracknet_csv, tracks_path)

run_player_extraction(
    video=str(local_video_path),
    output=str(players_path),
    model=PERSON_YOLO_MODEL,
    court_calibration=COURT_CALIBRATION or None,
    raw_output=str(player_raw_path),
)

read_track_csv(tracks_path, (video_info["width"], video_info["height"]), HeuristicConfig())
render_preview(local_video_path, tracks_path, player_raw_path, preview_path)
write_metadata(metadata_path, source_id, local_video_path, video_info, models)
zip_result_dir(result_dir, zip_path)

print("Artifacts:")
for path in [tracks_path, players_path, player_raw_path, metadata_path, preview_path, zip_path]:
    print(f"- {path}")


## 11. Export ZIP

Optionally copy the ZIP to Drive or download it through the browser.


In [ ]:
if COPY_ZIP_TO_DRIVE:
    drive_target = DRIVE_ROOT / "colab_outputs" / video_path.stem
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_target / zip_path.name)
    print(f"Copied ZIP to {drive_target / zip_path.name}")

if DOWNLOAD_ZIP_TO_BROWSER:
    from google.colab import files
    files.download(str(zip_path))
